# 模組 6：財報問答 RAG — 爬蟲抓真財報 → 加工 → 附引用問答、查無拒答

> 上一個 RAG 主題（課程 1 Lab3）你做過「最小 RAG」。今天升級成**對真實財報問答**：
> **用爬蟲（Lab1 那招）抓台積電的財報 → 加工成乾淨記錄 → RAG 附出處回答、查不到就拒答不瞎掰。**

> ## ⚠️ 一條工程紀律（先讀 30 秒）
>
> 今天用真實財報（台積電 2330，來源：富邦證券公開財報頁）純為教學示範 RAG，資料頁本身就標明「僅供參考、不負法律責任、投資人自負盈虧」。
> RAG 的答案只是**查證的起點**——就算附了引用，最終仍要「人」點回原始財報驗證。
> LLM 就算拿著正確資料，仍可能幻覺一個數字、或亂標引用——所以今天要教「**引用溯源 + 拒答**」兩道防線。

In [ ]:
# 📦 先跑這一格：指定版本，避免學校電腦裝到不相容的舊版（裝不起來看 README）
!pip install -q requests beautifulsoup4==4.12.3 langchain-ollama==1.0.1 langchain-community==0.4.2 faiss-cpu==1.14.2

## 🔧 第 0 步：環境就緒（沿用課程 1 那套）

In [ ]:
# ✅ 設定 API key（今天呼叫雲端生成的「門票」，沒有它 C 段做不下去）
import os

# 👇 同學：把引號中間換成老師給你的 OLLAMA key（整段貼進去，前後別留空白）
os.environ["OLLAMA_API_KEY"] = "在這裡貼上你的 OLLAMA key"

key = os.environ.get("OLLAMA_API_KEY")
if key and key != "在這裡貼上你的 OLLAMA key":
    print("✅ OLLAMA_API_KEY 已設定（長度", len(key), "個字元）")
else:
    print("❌ 還沒貼 key → 把上面那行引號中間換成你的 OLLAMA key，再重跑這一格")

# 🔒 只印長度、不印 key 本身——key 等於你帳號的鑰匙。貼了 key 的 notebook 別上傳 GitHub。

### 建立兩個模型物件（跟 Lab3 一模一樣，只差一個地方）

> `emb` 本地算向量（不花錢）、`llm` 雲端生成。**⚠️ 今天 `llm` 沒有 `format="json"`**——今天要的是「給人讀的自由文字 + 【資料n】標記」，不是 JSON。
> **預期輸出（示意・LLM 每次回答不同）：** 一句「我準備好了」→ 代表雲端可達。

In [1]:
from langchain_ollama import OllamaEmbeddings, ChatOllama

# (1) 本地 embedding —— Lab3 同一支：不帶 key、打 127.0.0.1:11434、不耗雲端額度
emb = OllamaEmbeddings(model="bge-m3")

# (2) 雲端生成 —— Lab3 同一支；⚠️ 沒有 format='json'（今天要自由文字+引用標記，不是 JSON）
llm = ChatOllama(
    model="gemma4:cloud",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"}},
)
print(llm.invoke("用一句繁體中文說你準備好了").content)   # 通了 → 雲端可達

我準備好了。


## 🟦 A 段：用爬蟲抓真財報（接 Lab1 金融爬蟲）

> 場景：老闆問「台積電上一季毛利率多少？跟前年比呢？」——你要能**查得動真財報**。
> 財報數字不是躺在一個乾淨檔案裡，是躺在**網頁表格**裡。所以今天第一步就是你 Lab1 學過的：**爬蟲抓下來**。
> 今天做三件事：**① 爬蟲抓財報網頁 → ② 加工成乾淨記錄 → ③ RAG 附引用問答、查無拒答。**

### A1・爬蟲抓台積電財報頁（富邦證券 djhtm）

> 抓兩張表：**綜合損益表**（絕對金額）＋**財務比率表**（毛利率/ROE/負債比率）。
> ⚠️ **這頁是 big5 編碼**（不是 utf-8）——抓下來要 `.decode("big5")`，不然中文會變亂碼。
> **預期輸出：** 兩張表的 HTML 長度各約五萬字元。

In [2]:
import requests, re

def crawl_djhtm(code):
    """抓一張富邦 djhtm 財報表（big5 編碼），回傳解碼後的 HTML 字串"""
    url = f"https://fubon-ebrokerdj.fbs.com.tw/z/zc/{code}/{code}_2330.djhtm"
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=20)
    return r.content.decode("big5", "ignore")     # ⭐ 這頁是 big5，不是 utf-8

income_html = crawl_djhtm("zcq")   # zcq = 綜合損益表（季）
ratio_html  = crawl_djhtm("zcr")   # zcr = 財務比率表（季）
print("損益表 HTML 長度：", len(income_html))
print("比率表 HTML 長度：", len(ratio_html))

損益表 HTML 長度： 49055
比率表 HTML 長度： 39633


### A2・用 bs4 解析：先看一列長怎樣

> 用 Lab1 學過的 **BeautifulSoup**：財報頁每一列是一個 `<div class="table-row">`、每格是 `<span class="table-cell">`。抓「營業毛利」那一列出來看。
> **整齊——但有個問題：** 數字（751,295…）**沒帶期別**，看不出哪個是哪一季。下一段「加工」就是把期別貼到每個數字上。

In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(income_html, "html.parser")
rows = soup.select("div.table-row")        # 每一列 = 一個科目
print("共", len(rows), "列")

# 抓「營業毛利」那一列，看它有哪些格
for r in rows:
    cells_row = [c.get_text(strip=True) for c in r.select("span.table-cell")]
    if cells_row and cells_row[0] == "營業毛利":
        print(cells_row)      # ['營業毛利', '751,295', '651,987', ...]：科目 + 8 個季度數字
        break

共 108 列
['營業毛利', '751,295', '651,987', '588,543', '547,369', '493,395', '512,378', '439,346', '358,124']


## 🟩 B 段：加工——把每一列變成乾淨記錄（今天的重點）

> **RAG 好不好用，八成決定在這一步。** bs4 給的是「一列 = 科目 + 8 個數字」，但**數字沒帶期別**。加工就是把它攤成一句一句**自帶期別的乾淨記錄**：
> `台積電(2330) 2025年第2季（2025Q2） 營業毛利：547,369`
> 每筆記錄自帶「表名 / 期別 / 科目」——這就是之後溯源的根。

### B1・加工：bs4 抓的列 → 一筆一句的乾淨記錄

> 把每一列攤成「一筆記錄 = 一個科目 × 一個季度」，並把期別貼上去。
> ⭐ **期別正規化**：把 `2024.3Q` 寫成 `2024年第3季（2024Q3）`——**兩種問法都撈得到**（不然問「2024第3季」會撈到「2025.3Q」，因為都有「3」）。
> **預期輸出：** 加工出 64 筆記錄。

In [4]:
from langchain_core.documents import Document

def normalize_period(p):
    """2024.3Q → 2024年第3季（2024Q3）：讓『2024第3季』和『2024Q3』兩種問法都撈得到"""
    year, q = p.split(".")
    q = q.replace("Q", "")
    return f"{year}年第{q}季（{year}Q{q}）"

def table_to_records(table_name, html, wanted):
    """把一張財報表 → 一筆一句的乾淨記錄：bs4 抓每一列，再把期別貼到每個數字上"""
    soup = BeautifulSoup(html, "html.parser")
    rows = soup.select("div.table-row")

    # 先找「期別」那一列，拿到 8 個季度的名稱
    periods = []
    for r in rows:
        cells_row = [c.get_text(strip=True) for c in r.select("span.table-cell")]
        if cells_row and cells_row[0] == "期別":
            periods = cells_row[1:9]
            break

    # 再把要的科目那幾列，攤成「一筆記錄 = 一個科目 × 一個季度」
    records = []
    for r in rows:
        cells_row = [c.get_text(strip=True) for c in r.select("span.table-cell")]
        if not cells_row:
            continue
        item = cells_row[0].replace("　", "")          # 第一格是科目名
        if item in wanted:
            for p, n in zip(periods, cells_row[1:9]):  # 後面 8 格對應 8 個季度
                q = normalize_period(p)
                records.append(Document(page_content=f"台積電(2330) {q} {item}：{n}",
                                        metadata={"表名": table_name, "期別": q, "科目": item}))
    return records

income_items = ["營業收入淨額", "營業毛利", "營業利益", "稅前淨利", "合併總損益", "每股盈餘"]
ratio_items  = ["營業毛利率", "營業利益率", "稅前純益率", "ROE(A)－稅後", "負債比率", "每股淨值(元)"]
records = table_to_records("綜合損益表", income_html, income_items)
records += table_to_records("財務比率表", ratio_html, ratio_items)

print(f"加工出 {len(records)} 筆乾淨記錄，樣本：")
for d in records[:3]:
    print("  ", d.page_content)

加工出 64 筆乾淨記錄，樣本：
   台積電(2330) 2026年第1季（2026Q1） 營業收入淨額：1,134,103
   台積電(2330) 2025年第4季（2025Q4） 營業收入淨額：1,046,090
   台積電(2330) 2025年第3季（2025Q3） 營業收入淨額：989,918


### 看一眼 records：每筆是一個 `Document`（page_content ＋ metadata）

> 上一格的 print 只看到文字；直接顯示 `records` 能看到完整的 `Document` 物件長相——**每筆都自帶 `表名 / 期別 / 科目`，這就是之後檢索、溯源都靠的結構。**

In [ ]:
records[:3]     # 看前 3 筆 Document 物件的完整長相

[Document(metadata={'表名': '綜合損益表', '期別': '2026年第1季（2026Q1）', '科目': '營業收入淨額'}, page_content='台積電(2330) 2026年第1季（2026Q1） 營業收入淨額：1,134,103'), Document(metadata={'表名': '綜合損益表', '期別': '2025年第4季（2025Q4）', '科目': '營業收入淨額'}, page_content='台積電(2330) 2025年第4季（2025Q4） 營業收入淨額：1,046,090'), Document(metadata={'表名': '綜合損益表', '期別': '2025年第3季（2025Q3）', '科目': '營業收入淨額'}, page_content='台積電(2330) 2025年第3季（2025Q3） 營業收入淨額：989,918')]

### B2・每筆記錄 = 一個檢索單位 → 建 FAISS 向量庫

> 因為加工後每筆已經是乾淨短句，**不用再切塊**（Lab3 的切塊是為了切長文件）。直接每筆記錄變一個向量存進 FAISS。
> **預期輸出：** 記錄數 64。

In [5]:
from langchain_community.vectorstores import FAISS

# 每一筆記錄就是一個檢索單位（乾淨短句，不需要再切塊）
vs = FAISS.from_documents(records, emb)
print("向量庫建好了，記錄數：", vs.index.ntotal)

向量庫建好了，記錄數： 64


### B3・`similarity_search_with_score`：撈回來的記錄「帶分數」

> `_with_score` 多回一個 **L2 距離**——**越小越像**（跟「相似度越大越像」相反）。
> **這個分數是 C 段拒答的溫度計**：撈回來全都很遠（分數全大）＝財報裡根本沒有相關內容。
> **預期輸出（示意・分數依 embedding 模型而定）：** 問「2025Q2 營業毛利」→ 對應那筆排最前、L2 約 0.4。

In [6]:
q = "台積電2025年第2季的營業毛利多少？"
scored = vs.similarity_search_with_score(q, k=3)   # 回 (記錄, L2距離)；越小越像

for d, s in scored:
    print(f"L2距離 {round(s, 3)}｜{d.metadata['表名']}·{d.metadata['期別']}｜{d.page_content}")

L2距離 0.405｜綜合損益表·2025年第2季（2025Q2）｜台積電(2330) 2025年第2季（2025Q2） 營業毛利：547,369
L2距離 0.437｜財務比率表·2025年第2季（2025Q2）｜台積電(2330) 2025年第2季（2025Q2） 營業毛利率：58.62
L2距離 0.481｜綜合損益表·2025年第2季（2025Q2）｜台積電(2330) 2025年第2季（2025Q2） 營業利益：463,424


## 🟧 C 段：引用溯源 + 兩道防線拒答 + 雙保險（**今天的課程重點**）

> 四步：① 附【資料n】引用 → ② 溯源回表名+期別 → ③ 查無拒答（兩道防線）→ ④ 雙保險（數字也要對得上）。
> 財報問答最怕 LLM 幻覺一個數字——所以答案要「有根、根要可查」。

### C1・把記錄編成【資料n】+ prompt 鐵則 → `ask_financials()`

> `build_context` 把記錄編成「【資料1】(綜合損益表·2025Q2) …」（**編號、表名、期別都是我們純 Python 編的、決定性**）；prompt 三條鐵則要求「只根據資料答、每句標【資料n】、查無就說查無」。
> **預期輸出（示意・LLM 每次措辭不同）：** 營業毛利 547,369，句末帶【資料1】。

In [7]:
def build_context(hits):
    """把檢索回來的 (記錄,分數) 編成【資料n】(表名·期別) 的 context——純 Python、決定性"""
    parts = []
    for i, (d, s) in enumerate(hits, 1):
        parts.append(f"【資料{i}】({d.metadata['表名']}·{d.metadata['期別']}) {d.page_content}")
    return "\n".join(parts)

RAG_PROMPT = """你是嚴謹的財報問答助理。鐵則：
1. 只能根據下面提供的資料回答，不可使用資料以外的知識。
2. 答案中每一句都要標注出處，格式為【資料n】。
3. 資料裡沒有的資訊，一律回「財報中查無此資訊」，絕不自己編。

【資料】
{context}

【問題】{q}
請用繁體中文回答："""

def ask_financials(question, k=3):
    """檢索 → 組帶引用的 context → prompt 鐵則 → 生成；多回傳 hits 供溯源"""
    hits = vs.similarity_search_with_score(question, k=k)
    ans = llm.invoke(RAG_PROMPT.format(context=build_context(hits), q=question)).content
    return ans, hits

ans, hits = ask_financials("台積電2025年第2季的營業毛利多少？")
print(ans)

台積電 2025 年第 2 季的營業毛利為 547,369【資料1】。


### C2・`trace_citations` 引用溯源（每個【資料n】→ 表名·期別 + 原文）

> **引用溯源（英文 citation / grounding，證照會考這兩個詞）**＝答案裡每個【資料n】都能反查回它指的那筆原文 + 來源，讓人「點一下就驗證」。
> 還有一個**防呆**：LLM 偶爾亂引一個不存在的【資料9】（只給了 3 筆）→ 超範圍編號當場標 ⚠️。**LLM 標了出處不代表出處存在，溯源函式才是裁判。**
> **預期輸出：** 真答案的【資料1】→ ✅ 綜合損益表·2025Q2；亂引【資料9】→ ⚠️。

In [8]:
def trace_citations(answer, hits):
    """從答案抽出所有【資料n】→ 還原成 (n, 表名·期別, 原文)；超範圍編號標 ⚠️ 亂引"""
    cited = [int(m) for m in re.findall(r"【資料(\d+)】", answer)]
    report = []
    for n in sorted(set(cited)):
        if 1 <= n <= len(hits):
            d = hits[n-1][0]        # 【資料n】的 n 是 build_context 我們自己編的 → 就是 hits 第 n 筆
            report.append((n, f"✅ {d.metadata['表名']}·{d.metadata['期別']}", d.page_content))
        else:
            report.append((n, "⚠️ 亂引（檢索結果沒有這個編號）", ""))
    return report

print(trace_citations(ans, hits))                                        # 對真答案溯源
fake_cite = "營業毛利 547,369 百萬元【資料1】，毛利率 58%【資料9】。"    # 只給 3 筆，哪來資料 9？
print(trace_citations(fake_cite, hits))

[(1, '✅ 綜合損益表·2025年第2季（2025Q2）', '台積電(2330) 2025年第2季（2025Q2） 營業毛利：547,369')]
[(1, '✅ 綜合損益表·2025年第2季（2025Q2）', '台積電(2330) 2025年第2季（2025Q2） 營業毛利：547,369'), (9, '⚠️ 亂引（檢索結果沒有這個編號）', '')]


### C3・查無就拒答：兩道防線（一道求 LLM、一道不求）

> 問一個**財報裡根本沒有的問題**（元宇宙部門營收），RAG 會怎樣？
> **第一道（prompt 鐵則）：** 檢索還是硬撈回來 → 送 LLM → 靠鐵則要它回「查無」。**多半聽話，但這是求它、沒保證。**
> **第二道（分數 gating）：** 撈回來 top-k 分數**全部超過門檻** → **根本不送 LLM、直接拒答**。純 Python 的 `if`、**決定性、還省一次雲端額度**。

In [9]:
# ── 第一道防線：prompt 鐵則（送 LLM、靠它聽話——非決定性）──
ans2, hits2 = ask_financials("台積電的元宇宙部門營收多少？")   # 財報根本沒有元宇宙部門
print(ans2)

財報中查無此資訊。


In [10]:
# ── 第二道防線：檢索分數 gating（根本不送 LLM——決定性邏輯）──
def retrieve_or_refuse(question, k=3, max_distance=0.85):
    """top-k 分數全部超過門檻 → 拒答不送 LLM。0.85＝這份財報語料的 bge-m3 校準值"""
    scored = vs.similarity_search_with_score(question, k=k)
    good = []
    for d, s in scored:
        if s <= max_distance:      # L2 距離越小越相關
            good.append((d, s))
    if not good:
        return None, scored        # 全部太遠 → 拒答，不送 LLM
    return good, scored

def ask_financials_safe(question, k=3, max_distance=0.85):
    """兩道防線合體版：先 gating（決定性），通過才送 LLM（仍掛 prompt 鐵則）"""
    good, scored = retrieve_or_refuse(question, k, max_distance)
    if good is None:
        return "財報查無相關資料（檢索距離全部超過門檻，未送 LLM）", scored
    ans = llm.invoke(RAG_PROMPT.format(context=build_context(good), q=question)).content
    return ans, good

print(ask_financials_safe("台積電的元宇宙部門營收多少？")[0])
print(ask_financials_safe("台積電2025年第2季的營業毛利多少？")[0])

財報查無相關資料（檢索距離全部超過門檻，未送 LLM）


台積電 2025 年第 2 季的營業毛利為 547,369【資料1】。


### C4・雙保險 `verify_cited_numbers`：答案裡的數字也要對得上

> 就算引用對了，LLM 仍可能在【資料1】旁邊幻覺一個數字。最後一道保險：**答案裡的每個數字，必須出現在「被引用的那幾筆記錄」裡**，查無就標 ⚠️。
> ⚠️ 財報數字有千分位逗號（547,369），比對前先去掉逗號。
> **預期輸出：** 正確的 547369 → ✅；故意餵的幻覺 999999 → ⚠️（被抓包）。

In [11]:
def verify_cited_numbers(answer, hits):
    """答案裡的每個數字，必須出現在被引用的記錄裡——財報數字有千分位逗號，先去掉再比對"""
    cited = set(int(m) for m in re.findall(r"【資料(\d+)】", answer))
    cited_text = ""
    for n in cited:
        if 1 <= n <= len(hits):
            cited_text += hits[n-1][0].page_content
    cited_text = cited_text.replace(",", "")
    clean = re.sub(r"【資料\d+】", "", answer).replace(",", "")     # 先去掉【資料n】再抓數字
    nums = re.findall(r"\d+(?:\.\d+)?", clean)
    result = {}
    for x in nums:
        result[x] = "✅在引用記錄中" if x in cited_text else "⚠️引用記錄查無(疑幻覺)"
    return result

print(verify_cited_numbers("台積電 2025Q2 營業毛利為 547,369 百萬元【資料1】。", hits))
print(verify_cited_numbers("台積電 2025Q2 營業毛利為 999,999 百萬元【資料1】。", hits))   # 999,999 是幻覺

{'2025': '✅在引用記錄中', '2': '✅在引用記錄中', '547369': '✅在引用記錄中'}
{'2025': '✅在引用記錄中', '2': '✅在引用記錄中', '999999': '⚠️引用記錄查無(疑幻覺)'}


### 📝 小作業

> 換一個**財報裡「有」的問題**（例：2025年第1季營業毛利率？／台積電負債比率？／2025Q4 稅前淨利？），用 `ask_financials_safe` 問一次，再跑 `trace_citations` 溯源 + `verify_cited_numbers` 驗數字。
>
> ⭐ **進階（選做）：** 把 `max_distance` 從 0.85 調小到 0.4，再問同一題——看它會不會「該答的被拒」，感受**門檻鬆緊的取捨**。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

- 問「2025年第1季營業毛利率是多少？」→ 財務比率表 2025Q1 營業毛利率 → 應答 **該季毛利率【資料n】**（例如約 58%，以當下爬到的數字為準）、`trace_citations` 指向 財務比率表·2025Q1、`verify_cited_numbers` 顯示數字在引用記錄中 ✅。
- ⭐ `max_distance=0.4`：這題 top1 L2 約 0.4 出頭 > 0.4 → 會被**誤拒**。說明門檻**太嚴會錯殺**；太鬆（如 1.0）則「員工人數」那種該拒的會漏放。**門檻沒有標準答案，要看 embedding 模型的分數分布校準**（我們在這份財報語料校在 0.85）。
</details>

In [12]:
# 📝 小作業參考解（參考解不只一種）
# 換一個財報裡「有」的問題，整條走一遍：拒答 gating → 附引用 → 溯源 + 數字雙保險
ans4, hits4 = ask_financials_safe("台積電2025年第1季的營業毛利率是多少？")
print("答案：", ans4)
print("溯源：", trace_citations(ans4, hits4))
print("數字查核：", verify_cited_numbers(ans4, hits4))
# 💡 財務比率表 2025Q1 營業毛利率 → 應答 XX%【資料n】、溯源指向 財務比率表·2025Q1、數字在引用記錄中 ✅

答案： 台積電2025年第1季的營業毛利率為58.79【資料1】。
溯源： [(1, '✅ 財務比率表·2025年第1季（2025Q1）', '台積電(2330) 2025年第1季（2025Q1） 營業毛利率：58.79')]
數字查核： {'2025': '✅在引用記錄中', '1': '✅在引用記錄中', '58.79': '✅在引用記錄中'}


## 🟫 D 段：收尾

```mermaid
flowchart LR
    WEB[富邦財報網頁<br/>損益表+財務比率] --> CR[爬蟲抓 big5 HTML<br/>Lab1 那招]
    CR --> TR[加工：表格→乾淨記錄<br/>期別正規化·附表名/期別]
    TR --> EM[本地 embedding bge-m3]
    EM --> VS[(FAISS 向量庫<br/>一筆記錄一個向量)]
    VS --> RET[帶分數檢索<br/>記錄+來源+L2分數]
    RET --> GATE{分數全部>門檻?}
    GATE -->|是| REF[拒答：查無相關資料<br/>不送 LLM·決定性]
    GATE -->|否| CTX[build_context<br/>【資料n】表名·期別]
    CTX --> LLM[雲端生成<br/>prompt 鐵則·附【資料n】]
    LLM --> TC[trace_citations<br/>溯源回表名·期別]
    TC --> VN[verify_cited_numbers<br/>數字雙保險]
    style GATE fill:#ffd6d6
    style TR fill:#fff2cc
    style TC fill:#ffe6cc
    style VN fill:#d4f1d4
```

> **一句話：** 爬蟲抓真財報 → **加工成乾淨記錄（今天的靈魂）** → 建庫 → 帶分數檢索 → gating 拒答 → 附引用回答 → 溯源 → 數字雙保險。

### 為什麼用「爬蟲抓網頁」而不是「解析 PDF」？（誠實說）

> 真財報 PDF（如證交所年報 PDF）拿去 RAG 會踩三個坑（實測過台積電官方財報 PDF）：
> - **有些頁抽出來是空的**（封面、簽章、圖片頁）
> - **表格會變數字湯**：數字有抽到，但行列對齊全毀 → LLM 對不上「哪個數字是哪一季」
> - **檢索門檻在上百頁時會崩**：什麼問題都能撈到「字面像」的一塊
>
> 改用「**爬蟲抓結構化網頁 + 加工成乾淨記錄**」就繞開全部——這也是為什麼今天**加工那一步才是重點**。真實 production RAG 的功夫，八成花在把髒資料變乾淨。

### 金融三鐵則收尾

> ① **數字回查原文**（今天＝引用溯源 + `verify_cited_numbers`；RAG 僅輔助、人做最後判斷）；② **資料不外傳**——誠實提醒：本課混合式會把財報數字送上雲端生成；真機敏內部文件要走「生成也換本地模型」的全本地路線。
> **🎓 證照接點：** RAG、向量資料庫、embedding、幻覺防範（grounding）、引用溯源（citation）都是 iPAS / AI-901 高頻考點。

## 🛟 Backup

> 爬蟲被擋 / 斷網時，改讀老師預存的 HTML（`data/` 內）。最後一格是「最小可跑核心」，純 Python、不需網路/faiss/Ollama，驗證可驗證鏈的骨幹還在。

In [13]:
# ── Backup①：爬蟲失敗時，改讀老師預存的 HTML（把下面兩行的 # 拿掉）──
# income_html = open("data/zcq_2330.html", "rb").read().decode("big5", "ignore")
# ratio_html  = open("data/zcr_2330.html", "rb").read().decode("big5", "ignore")

# ── Backup②：最小可跑核心（純 Python，不需網路/faiss/Ollama）──
import re

class Rec:   # 假記錄（只要有 page_content/metadata 就能餵下面三支函式）
    def __init__(self, text, table, period):
        self.page_content = text
        self.metadata = {"表名": table, "期別": period}

fake_hits = [
    (Rec("台積電(2330) 2025年第2季（2025Q2） 營業毛利：547,369", "綜合損益表", "2025年第2季（2025Q2）"), 0.55),
    (Rec("台積電(2330) 2025年第2季（2025Q2） 營業毛利率：58.62", "財務比率表", "2025年第2季（2025Q2）"), 0.61),
]

def build_context(hits):
    parts = []
    for i, (d, s) in enumerate(hits, 1):
        parts.append(f"【資料{i}】({d.metadata['表名']}·{d.metadata['期別']}) {d.page_content}")
    return "\n".join(parts)

def trace_citations(answer, hits):
    cited = [int(m) for m in re.findall(r"【資料(\d+)】", answer)]
    out = []
    for n in sorted(set(cited)):
        if 1 <= n <= len(hits):
            d = hits[n-1][0]
            out.append((n, f"✅ {d.metadata['表名']}·{d.metadata['期別']}", d.page_content))
        else:
            out.append((n, "⚠️ 亂引（檢索結果沒有這個編號）", ""))
    return out

def verify_cited_numbers(answer, hits):
    cited = set(int(m) for m in re.findall(r"【資料(\d+)】", answer))
    cited_text = ""
    for n in cited:
        if 1 <= n <= len(hits):
            cited_text += hits[n-1][0].page_content
    cited_text = cited_text.replace(",", "")
    clean = re.sub(r"【資料\d+】", "", answer).replace(",", "")
    nums = re.findall(r"\d+(?:\.\d+)?", clean)
    result = {}
    for x in nums:
        result[x] = "✅在引用記錄中" if x in cited_text else "⚠️引用記錄查無(疑幻覺)"
    return result

print(build_context(fake_hits)[:40], "...")
print(trace_citations("營業毛利 547,369【資料1】【資料9】。", fake_hits))     # 資料1 ✅、資料9 ⚠️亂引
print(verify_cited_numbers("營業毛利 547,369【資料1】。", fake_hits))         # 547369 ✅
print(verify_cited_numbers("營業毛利 999,999【資料1】。", fake_hits))         # 999999 ⚠️
good = []
for d, s in fake_hits:
    if s <= 0.85: good.append((d, s))
print("gating 通過筆數：", len(good))       # 2

【資料1】(綜合損益表·2025年第2季（2025Q2）) 台積電(2330)  ...
[(1, '✅ 綜合損益表·2025年第2季（2025Q2）', '台積電(2330) 2025年第2季（2025Q2） 營業毛利：547,369'), (9, '⚠️ 亂引（檢索結果沒有這個編號）', '')]
{'547369': '✅在引用記錄中'}
{'999999': '⚠️引用記錄查無(疑幻覺)'}
gating 通過筆數： 2
